# Polarization regulators heatmap
Minimal notebook to reproduce `polarization_regulators_heatmap.pdf`.

In [ ]:
# Imports
import os, sys
import pandas as pd
import numpy as np
import scanpy as sc
import pickle
import yaml

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from pert2state_model.Perturb2StateModel import Perturb2StateModel

sys.path.append('../3_DE_analysis/')
from DE_analysis_utils import get_DE_results_long

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['figure.autolayout'] = True
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['font.size'] = 16
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['ytick.labelsize'] = 15
plt.rcParams['legend.fontsize'] = 15

In [ ]:
# Load known regulators
with open('../../metadata/th1_th2_known_regulators.yaml', 'r') as f:
    known_regulators = yaml.safe_load(f)

regulator_type = {}
for cell_type, regulators in known_regulators.items():
    for reg in regulators:
        if cell_type == 'th1':
            regulator_type[reg] = 'Th1 regulator'
        elif cell_type == 'th2':
            regulator_type[reg] = 'Th2 regulator'

In [ ]:
# Load DE results AnnData
datadir = '/mnt/oak/users/emma/data/GWT/CD4i_final/'
experiment_name = 'CD4i_final'
adata_de = sc.read_h5ad(datadir + f'/DE_results_all_confounders/{experiment_name}.merged_DE_results.h5ad')

adata_de.layers['zscore'] = adata_de.layers['log_fc'] / adata_de.layers['lfcSE']
adata_de.layers['zscore'][np.where(adata_de.layers['zscore'] > 100)] = 100
adata_de.var_names = adata_de.var['gene_name'].values

# Filter very lowly expressed genes
base_mean_df = sc.get.obs_df(adata_de, adata_de.var_names.tolist() + ['culture_condition', 'chunk'], layer='baseMean')
base_mean_df = base_mean_df.drop_duplicates().sort_values('culture_condition')

base_mean_rest = base_mean_df[base_mean_df['culture_condition'] == 'Rest'].set_index('chunk').drop('culture_condition', axis=1).T
adata_de.var['mean_baseMean_Rest'] = base_mean_rest.mean(1).fillna(0)

base_mean_stim8 = base_mean_df[base_mean_df['culture_condition'] == 'Stim8hr'].set_index('chunk').drop('culture_condition', axis=1).T
adata_de.var['mean_baseMean_Stim8hr'] = base_mean_stim8.mean(1).fillna(0)

base_mean_stim48 = base_mean_df[base_mean_df['culture_condition'] == 'Stim48hr'].set_index('chunk').drop('culture_condition', axis=1).T
adata_de.var['mean_baseMean_Stim48hr'] = base_mean_stim48.mean(1).fillna(0)

gs_mask = (
    (adata_de.var['mean_baseMean_Rest'] > 0.1) &
    (adata_de.var['mean_baseMean_Stim8hr'] > 0.1) &
    (adata_de.var['mean_baseMean_Stim48hr'] > 0.1)
)
adata_de = adata_de[:, gs_mask].copy()

In [ ]:
# Load polarization signatures (Ota & Hollbacher)
ota_signature = pd.read_csv('results/Ota_Th2vsTh1_DE_results.csv', index_col='variable').dropna()
ota_signature.loc[ota_signature['zscore'] > 30, 'zscore'] = 30
ota_signature.loc[ota_signature['zscore'] < -30, 'zscore'] = -30

hollbacker_signature = pd.read_csv('results/hollbacher_Th2vsTh1_DE_results.csv', index_col='variable').dropna()
hollbacker_signature.loc[hollbacker_signature['zscore'] > 30, 'zscore'] = 30
hollbacker_signature.loc[hollbacker_signature['zscore'] < -30, 'zscore'] = -30

In [ ]:
# Extract top signature genes replicated in both cohorts
sig_genes_ota = ota_signature[ota_signature['adj_p_value'] < 0.01].index.tolist()
sig_genes_hollbacker = hollbacker_signature[hollbacker_signature['adj_p_value'] < 0.01].index.tolist()
sig_genes_both = list(set(sig_genes_ota) & set(sig_genes_hollbacker))

state_results_df = ota_signature[ota_signature.index.isin(sig_genes_both)].reset_index()

n_top = 50
signature_gs_up = state_results_df.nlargest(n_top, 'zscore').sort_values('zscore', ascending=False)['variable'].tolist()
signature_gs_down = state_results_df.nsmallest(n_top, 'zscore').sort_values('zscore')['variable'].tolist()

In [ ]:
# Load trained model and extract top/bottom regulators
with open('./results/polarization_prediction_condition_comparison.models.pkl', 'rb') as f:
    comparison_results = pickle.load(f)

c = 'Stim8hr'
dataset_key = f'ota_{c}'
model, X, y_train = comparison_results[dataset_key]
mod_coefs = model.get_coefs()

coef_mean = mod_coefs['coef_mean']
coef_ranks = coef_mean.rank(ascending=False)
n_coefs = len(coef_ranks)
mod_coefs['coef_rank'] = (coef_ranks - 1) / (n_coefs - 1)
mod_coefs['regulator'] = mod_coefs.index.str.split('_').str[0]

top_regs = mod_coefs.nlargest(20, 'coef_mean')
bottom_regs = mod_coefs.nsmallest(20, 'coef_mean')
all_regs = bottom_regs['regulator'].tolist() + top_regs['regulator'].tolist()

In [ ]:
# Build heatmap matrices
cytokines = ['IL13', 'IL5']
mod_coefs_indexed = model.get_coefs().copy()
mod_coefs_indexed.index = mod_coefs_indexed.index.str.split('_').str[0]

long_de_results = get_DE_results_long(
    adata_de,
    targets=all_regs,
    genes=signature_gs_up + signature_gs_down + cytokines,
    gene_id_col='gene_name'
)
long_de_results = long_de_results[long_de_results['culture_condition'] == c].copy()

wide_de_results = long_de_results.pivot(index='gene', columns='target_contrast_gene_name', values='zscore').T.fillna(0)
wide_de_results_signif = long_de_results.pivot(index='gene', columns='target_contrast_gene_name', values='adj_p_value').T

predicted_regulator = wide_de_results.index
row_annot = pd.DataFrame({'regression coef': mod_coefs_indexed.loc[predicted_regulator, 'coef_mean']})
col_annot = pd.DataFrame({'zscore': ota_signature.loc[wide_de_results.columns, 'zscore']})

In [ ]:
# Plot heatmap
cmap_row = plt.cm.PRGn
cmap_col = plt.cm.RdYlBu_r

vmax_row = max(abs(row_annot['regression coef'].min()), abs(row_annot['regression coef'].max()))
vmax_col = max(abs(col_annot['zscore'].min()), abs(col_annot['zscore'].max()))

row_colors = row_annot['regression coef'].apply(lambda x: cmap_row((x + vmax_row) / (2 * vmax_row)))
row_colors.index = predicted_regulator
col_colors = col_annot['zscore'].apply(lambda x: cmap_col((x + vmax_col) / (2 * vmax_col)))

g = sns.clustermap(
    wide_de_results,
    cmap='RdBu_r',
    center=0,
    row_cluster=True,
    col_cluster=True,
    xticklabels=True,
    yticklabels=True,
    method='ward',
    figsize=(20, 10),
    row_colors=[row_colors],
    col_colors=[col_colors],
    vmin=-5,
    vmax=5,
    cbar_pos=None,
)

# Overlay significance dots
row_order = g.dendrogram_row.reordered_ind
col_order = g.dendrogram_col.reordered_ind
for i, row_idx in enumerate(row_order):
    for j, col_idx in enumerate(col_order):
        if wide_de_results_signif.iloc[row_idx, col_idx] < 0.1:
            zscore = wide_de_results.iloc[row_idx, col_idx]
            dot_color = 'w' if abs(zscore) > 3 else 'k'
            g.ax_heatmap.plot(j + 0.5, i + 0.5, f'{dot_color}.', markersize=2)

plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90, ha='center', fontsize=14)
plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0, fontsize=14)
g.ax_heatmap.set_xlabel('Measured signature gene', fontsize=16)
g.ax_heatmap.set_ylabel('Perturbed regulator gene', fontsize=16)
g.ax_row_colors.set_xlabel('Predicted regulator\neffect ($w_r$)', fontsize=16, rotation=0, ha='right', va='top')
g.ax_col_colors.set_ylabel('Signature\nZ-score', fontsize=16, rotation=0, ha='right', va='bottom')

norm_row = plt.Normalize(vmin=-vmax_row, vmax=vmax_row)
norm_col = plt.Normalize(vmin=-vmax_col, vmax=vmax_col)
norm_heat = plt.Normalize(vmin=-5, vmax=5)

cax_row = g.fig.add_axes([0.2, 1.02, 0.1, 0.02])
plt.colorbar(plt.cm.ScalarMappable(norm=norm_row, cmap=cmap_row), cax=cax_row, orientation='horizontal')
cax_row.set_title('Predicted regulator\neffect ($w_r$)', fontsize=12, pad=10)

cax_col = g.fig.add_axes([0.4, 1.02, 0.1, 0.02])
plt.colorbar(plt.cm.ScalarMappable(norm=norm_col, cmap=cmap_col), cax=cax_col, orientation='horizontal')
cax_col.set_title('Th2/Th1 Signature\nZ-score', fontsize=12, pad=10)

cax_heat = g.fig.add_axes([0.6, 1.02, 0.1, 0.02])
plt.colorbar(plt.cm.ScalarMappable(norm=norm_heat, cmap='RdBu_r'), cax=cax_heat, orientation='horizontal')
cax_heat.set_title('Perturb-seq effect\nZ-score (Stim 8hr)', fontsize=12, pad=10)

plt.savefig('./results/polarization_regulators_heatmap.pdf', bbox_inches='tight')
plt.savefig('./results/polarization_regulators_heatmap.png', bbox_inches='tight')
plt.show()